In [ ]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import itertools

In [ ]:
num_class = [47, 10, 43, 10, 45, 196, 397, 10]
dataset_name = ["DTD", "EuroSAT", "GTSRB", "MNIST", "RESISC45", "Stanford_Cars", "SUN397", "SVHN"]
domain = "Base_Fine_Tuned"
transform_type = "Standard" # Base_Fine_Tuned_Classifier | Standard
model_name = "CLIP_ViT_Vision"
refining_type = "Standard" # Increment_Training | Standard
refiner = "Fine_Tuned" # Fine_Tuned | Reverse_Probe
indices = [i for i in range(12)]

In [ ]:
# Store all combinations of lengths 2 to 8
all_combinations = {}

for r in range(1, 9):  # lengths 2 through 8
    combos = list(itertools.combinations(dataset_name, r))
    all_combinations[r] = combos

## Loading of Matrices

In [ ]:
task_matrix = {}

for num_sets in range(1,9):
    task_matrix[num_sets] = {}
    for combo in range(len(all_combinations[num_sets])):
        matrix_name = ""
        for c in all_combinations[num_sets][combo]:
            matrix_name += f"{c}_"
        W_folder = f"../Data/Multi_Class_Augmentation/{refining_type}/{refiner}/Task_Matrices/{num_sets}/{domain}/{matrix_name}"
        task_matrix[num_sets][matrix_name] = {}
        for rep in range(1,6):
            task_matrix[num_sets][matrix_name][rep] = {"U": {}, "S": {}, "Vh": {}}
            W = np.load(f"{W_folder}/{rep}.npy")
            for i in indices:
                task_matrix[num_sets][matrix_name][rep]["U"][i], task_matrix[num_sets][matrix_name][rep]["S"][i], task_matrix[num_sets][matrix_name][rep]["Vh"][i] = np.linalg.svd(W[i])

In [ ]:
import scipy
from scipy.stats import sem

energy_capture = {}

for num_sets in range(1,9):
    energy_capture[num_sets] = {}
    for combo in range(len(all_combinations[num_sets])):
        matrix_name = ""
        for c in all_combinations[num_sets][combo]:
            matrix_name += f"{c}_"
        k_avg = []
        for rep in range(1,6):
            energy = task_matrix[num_sets][matrix_name][rep]["S"][11] ** 2
            cumulative = np.cumsum(energy)
            total = cumulative[-1]
            frac = cumulative / total

            threshold = 0.95
            k = np.searchsorted(frac, threshold) + 1
            print("k: {:0%} energy:".format(threshold), k)
            k_avg.append(k)
        k_mean = np.mean(k_avg)
        ci_low, ci_high = scipy.stats.t.interval(0.95, df=len(k_avg)-1, loc=k_mean, scale=sem(k_avg))
        print(f"{matrix_name} - Mean: {k_mean} +- {ci_high-k_mean}")
        energy_capture[num_sets][matrix_name] = (k_mean, ci_low, ci_high)

In [ ]:
for i in range(1,9):
    for combo in range(len(all_combinations[i])):
        matrix_name = ""
        for c in all_combinations[i][combo]:
            matrix_name += f"{c}_"
        print(energy_capture[i][matrix_name])

In [ ]:
import json

with open("k_95%_Task_Matrices.json", "w") as f:
    json.dump(energy_capture, f, indent=2)

## Graphing

In [ ]:
with open("k_95%_Task_Matrices.json", "r") as f:
    data = json.load(f)

In [ ]:
for i in range(1,9):
    for combo in range(len(all_combinations[i])):
        matrix_name = ""
        for c in all_combinations[i][combo]:
            matrix_name += f"{c}_"
        print(data[str(i)][matrix_name])

: 

## Old

In [ ]:
plt.figure(figsize=(8,4))
plt.plot(S[11][6], marker='.', linewidth=1)
plt.title('Singular values (linear scale)')
plt.xlabel('index i')
plt.ylabel(r'$\sigma_i$')
plt.grid(True)

plt.figure(figsize=(8,4))
plt.semilogy(S[11][6], marker='.', linewidth=1)
plt.title('Singular values (log scale)')
plt.xlabel('index i')
plt.ylabel(r'$\sigma_i$ (log scale)')
plt.grid(True)
plt.show()

In [ ]:
energy = S[11][6]**2
cumulative = np.cumsum(energy)
total = cumulative[-1]
frac = cumulative / total  # fraction of Frobenius energy captured

# Pick k for a threshold
threshold = 0.95
k = np.searchsorted(frac, threshold) + 1  # +1 because searchsorted returns idx
print("k for {:.0%} energy:".format(threshold), k)

# Plot fraction
plt.figure(figsize=(8,4))
plt.plot(frac, linewidth=2)
plt.axhline(threshold, color='red', linestyle='--')
plt.xlabel('k')
plt.ylabel('Fraction of energy captured')
plt.grid(True)
plt.show()

In [ ]:
# Operator Norm
print(np.linalg.norm(S[11][6], ord=2))